In [2]:
import os
import json
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal

from dotenv import load_dotenv
load_dotenv()

@dataclass(frozen=True)
class Provider:
    """One provider described as pure data"""
    
    name: str
    """Name of the provider"""
    env_var: str
    """Environment variable name for the provider's API key"""
    is_free: bool
    """Whether the provider is free or not"""
    base_url: str|None
    """Base URL for the provider's API"""
    model: str|None
    """Model name for the provider's API"""
    
    
PROVIDERS=[
    Provider(
        name="Groq",
        env_var="GROQ_API_KEY",
        is_free=False,
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-120b"  
    ),
    # can i use one more geoq provider -> answer: yes, you can add another provider by creating a new instance of the Provider class and adding it to the PROVIDERS dictionary. For example:
    Provider(
        name="Groq2",
        env_var="GROQ2_API_KEY",
        is_free=False,
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-20b"  
    ) 
]

def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var): # check if the environment variable for the provider's API key is set
            return provider
        
        expected = ", ".join(provider.env_var for provider in PROVIDERS)
        raise RuntimeError(f"No provider key set. Please set one of the following environment variables: {expected}")
        
def build_client(provider: Provider):
    from groq import Groq
    
    api_key = os.environ[provider.env_var]
    if provider.base_url is None:
        return Groq(api_key=api_key)
    else:
        return Groq(api_key=api_key, base_url=provider.base_url)
    
def have_any_key()->bool:
    return any(os.environ.get(provider.env_var) for provider in PROVIDERS)

def llm_reply(prompt: str, *, max_tokens: int = 400)->str:
    provider = select_provider()
    client = build_client(provider)
    response = client.chat_completion(
        model=provider.model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content 

In [4]:
WEATHER_DB = {
    "london": {"celcius": 22, "sky": "cloudy"},
    "paris": {"celcius": 24, "sky": "sunny"},
    "tallinn": {"celcius": 28, "sky": "sunny"},
    "moscow": {"celcius": 18, "sky": "rainy"},
    "tokyo": {"celcius": 30, "sky": "sunny"},
    "beijing": {"celcius": 26, "sky": "cloudy"},
    "new york": {"celcius": 28, "sky": "sunny"},
    "mumbai": {"celcius": 32, "sky": "rainy"},
    "cape town": {"celcius": 16, "sky": "cloudy"},
    "sydney": {"celcius": 20, "sky": "sunny"},
    "rio de janeiro": {"celcius": 22, "sky": "cloudy"},
    "cairo": {"celcius": 24, "sky": "sunny"},
    "mexico city": {"celcius": 20, "sky": "cloudy"},
}

In [5]:
def lookup_weather(location: str)->str:
    record = WEATHER_DB.get(location.lower())
    if record is None:
        return f"Sorry, I don't know the weather in {location}"
    return f"The weather in {location} is {record['celsius']} C and {record['sky']}."
    